# Fitness Survey OCR Pipeline (Gemini) - Elementary School form
# Nhớ kiểm tra folder success và fail hiện tại, nếu bạn sắp run lại hoàn toàn hoặc data khác, hãy xóa 2 file này đi
_by Hoang Dung

Local-only Jupyter notebook. No Colab / Google Drive dependency.

-> Need to self-config for successfully running with Google Colab


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install -q pymupdf pillow openpyxl pandas google-generativeai


In [ ]:
import os

# -- Original input folder (multi-page PDFs placed here by the user) ----------
PDF_INPUT_DIR        = "./input_red_pdfs"

# -- Split / working / result folders -----------------------------------------
SPLIT_INPUT_DIR      = "./input_splitted_red_pdfs"   # each page -> one PDF
FAILED_DIR           = "./failed_red_pdfs"            # pages that errored
SUCCESS_DIR          = "./success_red_pdfs"           # pages that succeeded

OUT_DIR              = "./ocr_red_output"
DPI                  = 300

# Please set your Gemini API key in the environment variable GEMINI_API_KEY before running this script OR set it here directly (not recommended for security reasons).

DEFAULT_GEMINI_API_KEY = ""  # <- Replace with your actual Gemini API key if you want to hardcode it (not recommended).
GEMINI_API_KEY = ""
MODEL_NAME = "gemini-3-flash-preview"  # <- Instead of "gemini-2.5-flash" with the same pricing [Dung confirmed, or you can check more on https://ai.google.dev/gemini-api/docs/pricing]! You can also use "gemini-3.5-turbo", "gemini-3.5-pro", etc,. for better performance, but it costs more.

os.makedirs(OUT_DIR,         exist_ok=True)
os.makedirs(PDF_INPUT_DIR,   exist_ok=True)
os.makedirs(SPLIT_INPUT_DIR, exist_ok=True)
os.makedirs(FAILED_DIR,      exist_ok=True)
os.makedirs(SUCCESS_DIR,     exist_ok=True)

if not GEMINI_API_KEY:
    print("Set GEMINI_API_KEY before running (see README).")
print(f"Model: {MODEL_NAME}")


In [ ]:
# All 68 output fields, in fixed order.
# Differences vs the junior-high (blue) form:
#  - No 持久走 test (elementary form only has 20mシャトルラン; 分/秒 fields do not exist)
#  - 8th test is ソフトボール投げ instead of ハンドボール投げ
#  - 質7 is a single yes/no field (地域のスポーツクラブに入っているか), not a 5-item checklist
#  - New 質7-2 field (frequency, only filled if 質7 = 入っている)
#  - 質8 (weekly activity time) is a single 7-day row, not 3 categories x 7 days
COLUMN_NAMES = [
    "学校名", "性別", "No.", "あく力_右", "あく力_左", "上体起こし", "長座体前くつ", "反復横とび",
    "20mシャトルラン", "50m走", "立ちはばとび", "ソフトボール投げ", "身長", "体重",
    "質1", "質2-❶", "質2-❷", "質2-❸", "質2-❹", "質2-❺", "質3",
    "質4-❶", "質4-❷", "質4-❸", "質4-❹", "質4-❺", "質4-❻その他", "質5",
    "質6-❶", "質6-❷", "質6-❸", "質7", "質7-2",
    "質8_月", "質8_火", "質8_水", "質8_木", "質8_金", "質8_土", "質8_日",
    "質8-2-①", "質8-2-②", "質8-2-③", "質8-2-④", "質8-2-⑤", "質8-2-⑤その他",
    "質9", "質10", "質11", "質12",
    "質12-2-①", "質12-2-②", "質12-2-③", "質12-2-④", "質12-2-⑤", "質12-2-⑥", "質12-2-⑦", "質12-2-⑧", "質12-2-⑨", "質12-2-⑩", "質12-2-⑩その他",
    "質13", "質14", "質15", "質16", "質17", "質18", "質19"
]
assert len(COLUMN_NAMES) == 68

# ROI groups matching physical form layout.
# Page = one spread: LEFT half (質8~19), RIGHT half (header + 実技 + 質1~7-2).
GROUP_HEADER_MEASURE = ["学校名","性別","No.","あく力_右","あく力_左","上体起こし","長座体前くつ","反復横とび",
                         "20mシャトルラン","50m走","立ちはばとび","ソフトボール投げ","身長","体重"]

GROUP_SURVEY_1_7 = ["質1","質2-❶","質2-❷","質2-❸","質2-❹","質2-❺","質3","質4-❶","質4-❷","質4-❸",
                     "質4-❹","質4-❺","質4-❻その他","質5","質6-❶","質6-❷","質6-❸",
                     "質7","質7-2"]

GROUP_CLUB_TIME = ["質8_月","質8_火","質8_水","質8_木","質8_金","質8_土","質8_日",
                    "質8-2-①","質8-2-②","質8-2-③","質8-2-④","質8-2-⑤","質8-2-⑤その他"]

GROUP_DAILY_HABIT = ["質9","質10","質11"]

GROUP_HEALTH_CLASS = ["質12","質12-2-①","質12-2-②","質12-2-③","質12-2-④","質12-2-⑤","質12-2-⑥",
                       "質12-2-⑦","質12-2-⑧","質12-2-⑨","質12-2-⑩","質12-2-⑩その他",
                       "質13","質14","質15","質16","質17","質18","質19"]

ALL_GROUPS = [
    ("header_measure", GROUP_HEADER_MEASURE),
    ("survey_1_7", GROUP_SURVEY_1_7),
    ("club_time", GROUP_CLUB_TIME),
    ("daily_habit", GROUP_DAILY_HABIT),
    ("health_class", GROUP_HEALTH_CLASS),
]
assert sorted(sum([g for _, g in ALL_GROUPS], [])) == sorted(COLUMN_NAMES)

# ROI boundaries as fraction of each half-page height, verified against actual scans
# with extra top/bottom padding so no field is ever clipped out of the crop.
ROI_FRACTIONS = {
    "header_measure": ("right", 0.00, 0.325),
    "survey_1_7":      ("right", 0.27, 1.00),
    "club_time":       ("left",  0.00, 0.20),
    "daily_habit":     ("left",  0.14, 0.335),
    "health_class":    ("left",  0.28, 1.00),
}

# FIELD_HINTS: real printed question text bound to each field key, so the model
# anchors on form content instead of guessing from surrounding rows/columns.
# "質8-2" and "質12-2" use a multi-column checkbox layout where the printed
# numbering goes left-to-right per row, then down (row-major) - NOT top-to-bottom
# per column. Mislabeling this order causes selected marks to land on the wrong key.
FIELD_HINTS = {
    "質1": "運動やスポーツをすることは好きですか。選択肢:好き/やや好き/やや嫌い/嫌い",
    "質2-❶": "❶運動やスポーツをすること。選択肢:ある/ややある/あまりない/ない",
    "質2-❷": "❷運動やスポーツをみること。選択肢:ある/ややある/あまりない/ない",
    "質2-❸": "❸運動やスポーツをささえること(大会運営のボランティアなど)。選択肢:ある/ややある/あまりない/ない",
    "質2-❹": "❹運動やスポーツを知ること(話を聞く、調べるなど)。選択肢:ある/ややある/あまりない/ない",
    "質2-❺": "❺運動やスポーツを通じていろいろな人が集まって交流したり、つながりや一体感を感じたりすること。選択肢:ある/ややある/あまりない/ない",
    "質3": "運動やスポーツをして、楽しいと感じますか。選択肢:感じる/やや感じる/あまり感じない/感じない",
    "質4-❶": "❶体を動かしてすっきりした気分になったとき。選択肢:感じる/やや感じる/あまり感じない/感じない",
    "質4-❷": "❷いろいろな種目を体験したとき。選択肢:感じる/やや感じる/あまり感じない/感じない",
    "質4-❸": "❸できなかったことができるようになったとき。選択肢:感じる/やや感じる/あまり感じない/感じない",
    "質4-❹": "❹記録に挑戦したり、記録があがったり、競い合ったりしたとき。選択肢:感じる/やや感じる/あまり感じない/感じない",
    "質4-❺": "❺友達と交流したり、協力できたとき。選択肢:感じる/やや感じる/あまり感じない/感じない",
    "質4-❻その他": "❻その他(自由記述の手書きテキストをそのまま転記。空欄ならnull)",
    "質5": "中学校に進んだら、授業以外でも自主的に運動やスポーツをする時間を持ちたいと思いますか。選択肢:思う/やや思う/あまり思わない/思わない",
    "質6-❶": "❶運動すること。選択肢:ある/ややある/あまりない/ない",
    "質6-❷": "❷スポーツの話をすること。選択肢:ある/ややある/あまりない/ない",
    "質6-❸": "❸スポーツ観戦をすること(テレビ観戦をふくむ)。選択肢:ある/ややある/あまりない/ない",
    "質7": "地域のスポーツクラブ(スポーツ少年団や習い事をふくみます)に入っていますか。選択肢:入っている/入っていない(単一選択、チェックボックスではない)",
    "質7-2": "質7で「入っている」と回答した人のみ対象。地域のスポーツクラブでの活動回数。"
             "選択肢:週1回/週2回/週3回/週4回/週5回/週6回/毎日。質7が「入っていない」ならnull",
    # 質8-2: 2-column x 2-row grid + その他, numbered row-major (left→right, then next row).
    "質8-2-①": "①運動する時間がないから(0/1、行1左)",
    "質8-2-②": "②運動する場所がないから(0/1、行1右)",
    "質8-2-③": "③一緒に運動する友達がいないから(0/1、行2左)",
    "質8-2-④": "④運動が好きではないから(0/1、行2右)",
    "質8-2-⑤": "⑤その他(0/1、最終行)。チェックされていれば右の自由記述欄に手書き理由があるはずなので、"
                "その内容は「質8-2-⑤その他」キーに転記すること",
    "質8-2-⑤その他": "質8-2⑤「その他」の横に書かれた自由記述欄の手書きテキスト。"
                    "⑤が0(未チェック)、または欄が空欄の場合はnull",
    "質9": "朝食は毎日食べますか。(学校が休みの日もふくめます)"
           "選択肢:毎日食べる/食べない日もある/食べない日が多い/食べない。"
           "選択されている選択肢の文言をそのまま値として出力してください(番号ではありません)。",
    "質10": "毎日どのくらい寝ていますか。"
            "選択肢:10時間以上/9時間以上10時間未満/8時間以上9時間未満/"
            "7時間以上8時間未満/6時間以上7時間未満/6時間未満。"
            "選択されている選択肢の文言をそのまま値として出力してください(番号ではありません)。",
    "質11": "平日(月〜金曜日)について。学習以外で、1日にどのくらいの時間、テレビやDVD、ゲーム機、"
            "スマートフォン、パソコンなどの画面を見ていますか。"
            "選択肢:5時間以上/4時間以上5時間未満/3時間以上4時間未満/2時間以上3時間未満/"
            "1時間以上2時間未満/1時間未満/全く見ない。"
            "選択されている選択肢の文言をそのまま値として出力してください(番号ではありません)。",
    "質12": "体育の授業は楽しいですか。選択肢:楽しい/やや楽しい/あまり楽しくない/楽しくない",
    # 質12-2: 2-column x 5-row grid + その他, numbered row-major (left→right, then next row).
    "質12-2-①": "①運動のポイントを分かりやすく教えてもらえたら(0/1、行1左)",
    "質12-2-②": "②できなかったことができるようになったら(0/1、行1右)",
    "質12-2-③": "③自分に合った場やルールが用意されていたら(0/1、行2左)",
    "質12-2-④": "④タブレットなどのICTを活用できたら(0/1、行2右)",
    "質12-2-⑤": "⑤先生にほめてもらえたら(0/1、行3左)",
    "質12-2-⑥": "⑥友達にみとめてもらえたら(0/1、行3右)",
    "質12-2-⑦": "⑦先生に個別に教えてもらえたら(0/1、行4左)",
    "質12-2-⑧": "⑧自分に合ったペースで行うことができたら(0/1、行4右)",
    "質12-2-⑨": "⑨できる・できないだけで比べられなかったら(0/1、行5左、右列なし)",
    "質12-2-⑩": "⑩その他(0/1、最終行)。チェックされていれば横の自由記述欄に手書き内容があるはずなので、"
                "その内容は「質12-2-⑩その他」キーに転記すること",
    "質12-2-⑩その他": "質12-2⑩「その他」の横に書かれた自由記述欄の手書きテキスト。"
                     "⑩が0(未チェック)、または欄が空欄の場合はnull",
    "質13": "体育の授業で運動やスポーツに取り組むときに、自分から「やってみたい」と思うときはありますか。"
            "選択肢:いつもある/だいたいある/あまりない/全くない",
    "質14": "目標(ねらい・めあて)を意識して学習することで「できたり、わかったり」することがありますか。同じ4択",
    "質15": "友達と助け合ったり、教え合ったりして学習することで「できたり、わかったり」することがありますか。同じ4択",
    "質16": "うまくいかないことがあったときにも、あきらめずに取り組めましたか。選択肢:いつも取り組めていた/だいたい取り組めていた/あまり取り組めていなかった/取り組めていなかった",
    "質17": "タブレットなどのICTを使って学習することで「できたり、わかったり」することがありますか。選択肢:いつもある/だいたいある/あまりない/全くない/ICTを活用していない",
    "質18": "保健の授業で学習した運動、食事、休養、すいみんに気をつけた生活を送れていると思いますか。選択肢:思う/やや思う/あまり思わない/思わない",
    "質19": "保健を学習して、もっと運動しようと思いましたか。選択肢:思うようになった/やや思うようになった/あまり思わなかった/思わなかった",
    "あく力_右": "①あく力・右。単位 kg",
    "あく力_左": "①あく力・左。単位 kg",
    "上体起こし": "②上体起こし。単位 回",
    "長座体前くつ": "③長座体前くつ。単位 cm",
    "反復横とび": "④反復横とび。単位は「点(回)」",
    "20mシャトルラン": "⑤20mシャトルラン。単位 回(この学年に持久走の分・秒フィールドは存在しない)",
    "50m走": "⑥50m走。単位 秒",
    "立ちはばとび": "⑦立ちはばとび。単位 cm",
    "ソフトボール投げ": "⑧ソフトボール投げ。単位 m",
    "身長": "(1)身長。単位は「cm」。小数第1位まで",
    "体重": "(2)体重。単位は「kg」。小数第1位まで",
}

# Groups containing a multi-column (row-major) checkbox grid.
GRID_CHECKLIST_GROUPS = {"club_time", "health_class"}

# Groups containing a "その他" checkbox paired with a free-text field.
TEXT_COMPANION_GROUPS = {"survey_1_7", "club_time", "health_class"}


In [ ]:
import fitz
import numpy as np
from PIL import Image

def render_page(pdf_path, page_num, dpi=DPI):
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    zoom = dpi / 72
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()
    return img

def crop_roi(full_page_img, side, frac_top, frac_bottom):
    """side: 'left' or 'right' half of the page; frac_top/bottom: height ratio (0-1)."""
    w, h = full_page_img.size
    x0, x1 = (0, w // 2) if side == "left" else (w // 2, w)
    y0, y1 = int(h * frac_top), int(h * frac_bottom)
    return full_page_img.crop((x0, y0, x1, y1))

def get_all_rois(full_page_img):
    rois = {}
    for group_name, (side, top, bot) in ROI_FRACTIONS.items():
        rois[group_name] = crop_roi(full_page_img, side, top, bot)
    return rois


In [ ]:
import io, json, time
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

def build_schema(fields):
    return {
        "type": "object",
        "properties": {f: {"type": "string", "nullable": True} for f in fields},
        "required": fields,
    }

def build_prompt(fields, group_name):
    hint_lines = [f'- "{f}": {FIELD_HINTS[f]}' for f in fields if f in FIELD_HINTS]
    hint_block = ""
    if hint_lines:
        hint_block = "各キーが対応する設問文（フォーム上の印刷文）:\n" + "\n".join(hint_lines) + "\n"

    anti_shift = ""
    if group_name in ("survey_1_7", "health_class"):
        anti_shift = (
            "【重要・厳守】各設問は横一列に4つの選択肢が並んでいます。"
            "必ず「この設問番号の行」に実際に塗られている／マークされている選択肢だけを読み取り、"
            "前後の設問のパターンから類推して埋めないでください。設問ごとに独立して判定してください。\n"
        )
        if group_name == "survey_1_7":
            anti_shift += (
                "【注意】「あまりない」(4文字)と「ない」(2文字)は末尾が共通しています。"
                "判定する前に、「ない」の直前に「あまり」の2文字が存在するか必ず確認してください。"
                "存在する場合は「あまりない」、空白や塗りつぶしマークの直後に「ない」がある場合は「ない」を出力してください。\n"
            )

    grid_order = ""
    if group_name in GRID_CHECKLIST_GROUPS:
        grid_order = (
            "【重要・厳守】チェックリスト形式の設問(質8-2、質12-2)は2列レイアウトです。"
            "番号は「左列から右列へ、その後次の行へ」の順（行優先）で振られています。"
            "列ごとに上から下へ読み進めないでください。各キーに紐づく設問文(上記)と"
            "画像上の実際の印刷テキストを照合し、正しい行・列のチェック有無だけを読み取ってください。\n"
        )

    single_select_note = ""
    if group_name == "survey_1_7":
        single_select_note = (
            "質7は単一選択です(入っている／入っていない)。チェックボックスの複数選択ではありません。"
            "質7-2は質7が「入っている」の場合のみ値を入れ、「入っていない」ならnullにしてください。\n"
        )

    text_companion = ""
    if group_name in TEXT_COMPANION_GROUPS:
        text_companion = (
            "【重要】「その他」のチェック欄の横または下には、手書きの自由記述欄があります。"
            "対応するチェックが1(あり)の場合、その手書きテキストを対応する「...その他」キーに"
            "そのまま転記してください。小さい文字も見落とさず確認してください。"
            "チェックが0、または記述欄が空欄の場合はnullとしてください。\n"
        )

    return f"""あなたはOCRのプロフェッショナルです。添付画像から次のキーの値を正確に読み取り、JSONのみ返してください。
読み取れない・空欄は null。
{hint_block}{anti_shift}{grid_order}{single_select_note}{text_companion}
キー一覧: {json.dumps(fields, ensure_ascii=False)}"""

def call_gemini_group(pil_image, fields, group_name, temperature=0.0):
    model = genai.GenerativeModel(MODEL_NAME)
    prompt = build_prompt(fields, group_name)
    resp = model.generate_content(
        [prompt, pil_image],
        generation_config={
            "temperature": temperature,
            "response_mime_type": "application/json",
            "response_schema": build_schema(fields),
        },
        request_options={"timeout": 4000}
    )
    time.sleep(2)  # basic rate-limit throttle
    return json.loads(resp.text)

def call_group_self_consistent(pil_image, fields, group_name):
    """Single-call mode (self-consistency double-check disabled to save cost)."""
    r1 = call_gemini_group(pil_image, fields, group_name, temperature=0.0)
    final, flags = {}, {}
    for f in fields:
        final[f] = r1.get(f)
        flags[f] = "OK"
    return final, flags


## Step 1 - Split every PDF in `PDF_INPUT_DIR` into single-page PDFs -> `SPLIT_INPUT_DIR`

Each page is extracted as an independent PDF (no image conversion, no quality loss).  
Re-running this cell is safe: files that already exist in `SPLIT_INPUT_DIR` are skipped.


In [ ]:
import glob, shutil

source_pdfs = sorted(glob.glob(os.path.join(PDF_INPUT_DIR, '*.pdf')))
print(f"Found {len(source_pdfs)} PDF file(s) in {PDF_INPUT_DIR}")

split_count = 0
for src_path in source_pdfs:
    src_name = os.path.splitext(os.path.basename(src_path))[0]
    doc = fitz.open(src_path)
    n_pages = len(doc)
    for page_num in range(n_pages):
        # Naming: <original_stem>_p{page_num:04d}.pdf
        out_name = f"{src_name}_p{page_num:04d}.pdf"
        out_path = os.path.join(SPLIT_INPUT_DIR, out_name)
        if os.path.exists(out_path):
            continue  # already split - skip
        single = fitz.open()          # new empty PDF
        single.insert_pdf(doc, from_page=page_num, to_page=page_num)
        single.save(out_path, garbage=4, deflate=True)  # lossless page copy
        single.close()
        split_count += 1
    doc.close()

all_split_files = sorted(glob.glob(os.path.join(SPLIT_INPUT_DIR, '*.pdf')))
print(f"Split {split_count} new page(s). Total in {SPLIT_INPUT_DIR}: {len(all_split_files)} file(s).")


## Step 2 - Run pipeline on all single-page PDFs in `SPLIT_INPUT_DIR`

* **Success** -> record saved + file moved to `SUCCESS_DIR`  
* **Error (any exception, e.g. 429/503/504)** -> file moved to `FAILED_DIR`, pipeline continues with next file


In [ ]:
import glob, shutil, traceback

def run_pipeline_on_dir(input_dir, all_final, all_flags_rows):
    """
    Process every single-page PDF found in *input_dir*.
    Successful files are moved to SUCCESS_DIR; failed files to FAILED_DIR.
    Results are appended in-place to all_final and all_flags_rows.
    Returns (n_success, n_failed).
    """
    pdf_files = sorted(glob.glob(os.path.join(input_dir, '*.pdf')))
    print(f"Found {len(pdf_files)} PDF file(s) in {input_dir}")

    n_success = 0
    n_failed  = 0

    for pdf_path in pdf_files:
        fname = os.path.basename(pdf_path)
        t0 = time.time()
        try:
            # Each split PDF has exactly 1 page (page 0)
            full_img = render_page(pdf_path, 0)
            rois = get_all_rois(full_img)

            record, flag_record = {}, {}
            for group_name, fields in ALL_GROUPS:
                final, flags = call_group_self_consistent(rois[group_name], fields, group_name)
                record.update(final)
                flag_record.update(flags)

            all_final.append(record)
            all_flags_rows.append(flag_record)

            elapsed = time.time() - t0
            print(f"  [OK]   {fname}  No.={record.get('No.')}  ({elapsed:.1f}s)")

            # Move to success folder
            shutil.move(pdf_path, os.path.join(SUCCESS_DIR, fname))
            n_success += 1

        except Exception as exc:
            elapsed = time.time() - t0
            print(f"  [FAIL] {fname}  ({elapsed:.1f}s)  -> {type(exc).__name__}: {exc}")
            traceback.print_exc()

            # Move to failed folder
            shutil.move(pdf_path, os.path.join(FAILED_DIR, fname))
            n_failed += 1

    return n_success, n_failed


# -- Main run ------------------------------------------------------------------
all_final      = []
all_flags_rows = []

print("=" * 60)
print("PASS 1 - processing split PDFs")
print("=" * 60)
n_ok1, n_fail1 = run_pipeline_on_dir(SPLIT_INPUT_DIR, all_final, all_flags_rows)
print(f"\nPass 1 done. Success: {n_ok1}, Failed: {n_fail1}")


## Step 3 - Export results to Excel (Pass 1) + Retry failed files (Pass 2)


In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font

def export_to_excel(records, flags_rows, label=''):
    """Save current results to a timestamped Excel file. Returns the saved path."""
    if not records:
        print(f"[Export{label}] No records to export - skipping.")
        return None

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    out_path = os.path.join(OUT_DIR, f"体力測定調査結果_小学校_{MODEL_NAME}_{timestamp}{label}.xlsx")

    df_final = pd.DataFrame(records).reindex(columns=COLUMN_NAMES)
    df_flags = pd.DataFrame(flags_rows).reindex(columns=COLUMN_NAMES)

    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    def append_df(wb, df, name):
        ws = wb.create_sheet(title=name)
        ws.append(list(df.columns))
        for row in df.fillna('').values.tolist():
            ws.append(row)
        return ws

    ws_final = append_df(wb, df_final, "納品データ")
    ws_flags = append_df(wb, df_flags, "自己一致性チェック")

    red_fill = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    red_font = Font(color='9C0006', bold=True)
    for r in range(2, len(df_final) + 2):
        for c, col in enumerate(COLUMN_NAMES, start=1):
            if ws_flags.cell(row=r, column=c).value == 'MISMATCH':
                cell = ws_final.cell(row=r, column=c)
                cell.fill = red_fill
                cell.font = red_font

    wb.save(out_path)
    print(f"Saved: {out_path}")
    return out_path


# -- Export Pass-1 results -----------------------------------------------------
print("=" * 60)
print("EXPORT 1 - results from Pass 1")
print("=" * 60)
export_to_excel(all_final, all_flags_rows, label='_pass1')


# -- Retry failed files (Pass 2) -----------------------------------------------
failed_files = sorted(glob.glob(os.path.join(FAILED_DIR, '*.pdf')))

if not failed_files:
    print("\nNo failed files - nothing to retry. All done!")
else:
    print(f"\n{'=' * 60}")
    print(f"PASS 2 - retrying {len(failed_files)} failed file(s)")
    print("=" * 60)

    all_final_retry      = []
    all_flags_rows_retry = []

    n_ok2, n_fail2 = run_pipeline_on_dir(FAILED_DIR, all_final_retry, all_flags_rows_retry)
    print(f"\nPass 2 done. Success: {n_ok2}, Still failed: {n_fail2}")

    # Export Pass-2 results (only records that came from the retry)
    print("\n" + "=" * 60)
    print("EXPORT 2 - results from Pass 2 (retry)")
    print("=" * 60)
    export_to_excel(all_final_retry, all_flags_rows_retry, label='_pass2_retry')

    remaining = sorted(glob.glob(os.path.join(FAILED_DIR, '*.pdf')))
    if remaining:
        print(f"\n[WARNING] {len(remaining)} file(s) still in {FAILED_DIR} after retry:")
        for f in remaining:
            print(f"  - {os.path.basename(f)}")
    else:
        print(f"\nAll retried files succeeded. {FAILED_DIR} is now empty.")

print("\n=== Pipeline finished ===")
